# 🏦 Anti-Money Laundering (AML) Detection Pipeline
### End-to-End Machine Learning Project

---

**Dataset:** PaySim – Synthetic Financial Datasets For Fraud Detection (Kaggle)  
**Pipeline:** Preprocessing → PCA → K-Means Clustering → XGBoost → Continuous Learning

---

## 📦 Install & Import Libraries

We only use what we need. Each library has one job:
- `pandas` → handle data tables
- `numpy` → math operations
- `matplotlib / seaborn` → simple charts
- `sklearn` → PCA, K-Means, scaling, metrics
- `xgboost` → our fraud detection classifier

In [ ]:
# Install xgboost if not already installed
# !pip install xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

print("✅ All libraries imported successfully!")

---
## 📂 Load the Dataset

Download the PaySim dataset from Kaggle:  
👉 https://www.kaggle.com/datasets/ealaxi/paysim1

Save the file as `paysim.csv` in the same folder as this notebook.

In [ ]:
# Load the dataset
df = pd.read_csv('paysim.csv')

# Quick look at the data
print("Shape:", df.shape)
df.head()

In [ ]:
# Check column names and data types
df.info()

In [ ]:
# Check how many fraud vs normal transactions exist
print("Class distribution (isFraud):")
print(df['isFraud'].value_counts())
print()
print(f"Fraud rate: {df['isFraud'].mean()*100:.2f}%")

---
## 🔹 Phase 1: Data Preprocessing & Feature Engineering

**What & Why:**  
Raw data is messy. Machine learning models only understand numbers, not text like "TRANSFER" or "CASHOUT". We also need all numbers to be on a similar scale so that large values don't dominate the model.

Steps:
1. Drop columns we don't need (like account names)
2. Check for missing values
3. Encode the `type` column (text → number)
4. Scale the numerical columns

In [ ]:
# --- Step 1: Drop unnecessary columns ---
# Account names (nameOrig, nameDest) are random IDs with no predictive value
# isFlaggedFraud is a rule-based flag we don't want to use as a feature
df = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])

print("Remaining columns:", df.columns.tolist())

In [ ]:
# --- Step 2: Check for missing values ---
print("Missing values per column:")
print(df.isnull().sum())

# PaySim is clean — no missing values expected
# If there were missing values, we'd fill numeric ones with the median:
# df['amount'].fillna(df['amount'].median(), inplace=True)

In [ ]:
# --- Step 3: Encode the 'type' column (text → number) ---
# The 'type' column has values like: TRANSFER, CASHOUT, PAYMENT, etc.
# We use Label Encoding to convert each unique type to an integer

le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

print("Transaction types encoded:")
for i, name in enumerate(le.classes_):
    print(f"  {name} → {i}")

In [ ]:
# --- Step 4: Separate features and target label ---
# 'isFraud' is what we want to predict (the label)
# Everything else is input data (features)

X = df.drop(columns=['isFraud'])   # Features
y = df['isFraud']                  # Target

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# --- Step 5: Scale the numerical features ---
# StandardScaler makes each column have mean=0 and standard deviation=1
# This prevents large numbers (like amount) from dominating the model

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to a DataFrame so we keep column names
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("✅ Data scaled. Sample means (should be close to 0):")
print(X_scaled.mean().round(3))

---
## 🔹 Phase 2: Dimensionality Reduction with PCA

**What & Why:**  
We have many features, and some of them might be telling us similar information (correlated). PCA compresses these into fewer "super-features" called principal components, while keeping most of the information.

Think of it like summarizing a long document into bullet points — you keep the key ideas and throw away the repetition.

We use a **scree plot** to decide how many components to keep.

In [ ]:
# --- Apply PCA to all scaled features ---
# First, we run PCA with ALL components to see how much variance each one explains

pca_full = PCA()
pca_full.fit(X_scaled)

# Cumulative explained variance: how much total info is captured as we add more components
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# --- Plot the Scree Plot ---
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, marker='o', color='steelblue')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance threshold')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA – How Many Components Do We Need?')
plt.legend()
plt.tight_layout()
plt.show()

# Print how many components we need to reach 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Components needed to explain 95% of variance: {n_components_95}")

In [ ]:
# --- Apply PCA with the chosen number of components ---
# We keep enough components to explain 95% of the variance

n_components = n_components_95

pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)

# Convert to DataFrame
pca_cols = [f'PC{i+1}' for i in range(n_components)]
X_pca_df = pd.DataFrame(X_pca, columns=pca_cols)

print(f"✅ PCA applied. Data shape reduced to: {X_pca_df.shape}")
print(f"   Total variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

---
## 🔹 Phase 3: Behavioral Segmentation with K-Means Clustering

**What & Why:**  
Before labeling transactions as fraud, we want to understand *behavioral patterns*. K-Means groups similar transactions together into clusters. 

Fraudulent transactions often cluster separately from normal ones — so the cluster label becomes a useful signal for our classifier.

We use the **Elbow Method** to find the best number of clusters (`k`).

In [ ]:
# --- Elbow Method: Find the optimal number of clusters ---
# We try k from 2 to 10 and measure inertia (how tight the clusters are)
# The "elbow" in the plot is where adding more clusters stops helping much

inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(X_pca_df)
    inertia_values.append(kmeans_temp.inertia_)
    print(f"  k={k} → inertia={kmeans_temp.inertia_:.0f}")

# --- Plot the Elbow Curve ---
plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia_values, marker='o', color='darkorange')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.title('Elbow Method – Finding the Optimal k')
plt.tight_layout()
plt.show()

In [ ]:
# --- Apply K-Means with the chosen k ---
# Pick k based on the elbow plot above (often 3 or 4 works well for PaySim)

k_chosen = 3  # ← Adjust this based on what the elbow plot shows

kmeans = KMeans(n_clusters=k_chosen, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_pca_df)

print(f"✅ K-Means done with k={k_chosen}")
print("Cluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
# --- Add cluster labels back to the original feature set ---
# This is the new engineered feature: behavioral_segment

X_scaled['behavioral_segment'] = cluster_labels

print("✅ 'behavioral_segment' feature added to dataset.")
print(X_scaled[['behavioral_segment']].head())

In [ ]:
# --- (Optional) Visualize clusters using first 2 PCA components ---
plt.figure(figsize=(8, 5))
scatter = plt.scatter(X_pca_df['PC1'], X_pca_df['PC2'],
                      c=cluster_labels, cmap='Set1', alpha=0.3, s=5)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title(f'K-Means Clusters (k={k_chosen}) on PCA Space')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

---
## 🔹 Phase 4: Fraud Detection with XGBoost

**What & Why:**  
XGBoost is a powerful algorithm that builds many decision trees and combines them to make accurate predictions. It's widely used in finance because it handles imbalanced data well and is very fast.

We use `scale_pos_weight` to help the model pay more attention to the rare fraud cases.

We evaluate using **Precision, Recall, and F1-score** — NOT accuracy, because with only 0.1% fraud, a model that predicts "no fraud" for everything would get 99.9% accuracy but be completely useless.

In [ ]:
# --- Split data into training (80%) and testing (20%) ---
from sklearn.model_selection import train_test_split

X_final = X_scaled.copy()  # This now includes behavioral_segment
y_final = y.reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=0.2,
    random_state=42,
    stratify=y_final  # Keep same fraud ratio in both splits
)

print(f"Training size: {X_train.shape[0]} rows")
print(f"Testing size:  {X_test.shape[0]} rows")
print(f"Fraud in training: {y_train.sum()} ({y_train.mean()*100:.2f}%)")

In [ ]:
# --- Handle class imbalance with scale_pos_weight ---
# Fraud is rare. We tell XGBoost to weight fraud cases more heavily.
# scale_pos_weight = (number of normal) / (number of fraud)

normal_count = (y_train == 0).sum()
fraud_count  = (y_train == 1).sum()
scale_weight = normal_count / fraud_count

print(f"Normal transactions: {normal_count}")
print(f"Fraud transactions:  {fraud_count}")
print(f"scale_pos_weight:    {scale_weight:.1f}")

In [ ]:
# --- Train the XGBoost Model ---
model = XGBClassifier(
    n_estimators=100,           # Number of trees
    max_depth=5,                # How deep each tree can go
    learning_rate=0.1,          # How fast the model learns
    scale_pos_weight=scale_weight,  # Handle imbalance
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)
print("✅ Model trained!")

In [ ]:
# --- Evaluate the model on the test set ---
y_pred = model.predict(X_test)

print("=" * 50)
print("Classification Report:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

In [ ]:
# --- Confusion Matrix ---
# Rows = actual label, Columns = predicted label
# Top-left: correctly predicted normal (True Negatives)
# Bottom-right: correctly predicted fraud (True Positives)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Normal', 'Predicted Fraud'],
            yticklabels=['Actual Normal', 'Actual Fraud'])
plt.title('Confusion Matrix – XGBoost Fraud Detection')
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Importance ---
# Which features did the model find most useful?

feat_importance = pd.Series(model.feature_importances_, index=X_final.columns)
feat_importance = feat_importance.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feat_importance.plot(kind='barh', color='steelblue')
plt.title('Feature Importance – XGBoost')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## 🔹 Phase 5: Continuous Learning (Batch/Partial Training)

**What & Why:**  
Money launderers adapt their behavior over time. A model trained once will become outdated. We simulate a scenario where **new transaction data arrives the following month**.

**The rule:** We are NOT allowed to mix old and new data and retrain from scratch. We must update the model using **only the new data** — this is called *warm-start* or *incremental training*.

With XGBoost, we do this using `xgb_model` parameter to continue training from existing model weights.

In [ ]:
# --- Simulate: New month's data arrives ---
# We pretend the last 10% of training data is "new data from next month"
# In a real scenario, this would be a fresh CSV file of new transactions

split_point = int(len(X_train) * 0.9)

X_new_batch = X_train.iloc[split_point:]   # New incoming transactions
y_new_batch = y_train.iloc[split_point:]

print(f"New batch size: {X_new_batch.shape[0]} transactions")
print(f"Fraud in new batch: {y_new_batch.sum()}")

In [ ]:
# --- Update the existing model with only the new batch ---
# We pass xgb_model=model to continue training from where we left off
# This is true partial/incremental training — no old data used!

import xgboost as xgb

# Convert data to DMatrix format (XGBoost's internal format)
dtrain_new = xgb.DMatrix(X_new_batch, label=y_new_batch)
dtest      = xgb.DMatrix(X_test, label=y_test)

# Convert existing sklearn model to booster (XGBoost's native model)
booster = model.get_booster()

# Set training parameters
params = {
    'max_depth': 5,
    'learning_rate': 0.1,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'scale_pos_weight': scale_weight
}

# Continue training for 20 more rounds on new data only
updated_booster = xgb.train(
    params=params,
    dtrain=dtrain_new,
    num_boost_round=20,      # Only 20 new trees added
    xgb_model=booster        # ← Start from existing model!
)

print("✅ Model updated with new batch (no old data used!)")

In [ ]:
# --- Evaluate the updated model ---
# Get predictions from the updated booster
y_pred_updated_proba = updated_booster.predict(dtest)
y_pred_updated = (y_pred_updated_proba >= 0.5).astype(int)

print("=" * 50)
print("Updated Model – Classification Report:")
print("=" * 50)
print(classification_report(y_test, y_pred_updated, target_names=['Normal', 'Fraud']))

In [ ]:
# --- Compare: Before vs After Update ---
from sklearn.metrics import f1_score, recall_score, precision_score

print("📊 Performance Comparison (Fraud class only):")
print("-" * 40)
print(f"{'Metric':<15} {'Before Update':>15} {'After Update':>15}")
print("-" * 40)

for metric_name, metric_fn in [('Precision', precision_score), ('Recall', recall_score), ('F1-Score', f1_score)]:
    before = metric_fn(y_test, y_pred)
    after  = metric_fn(y_test, y_pred_updated)
    print(f"{metric_name:<15} {before:>15.3f} {after:>15.3f}")

---
## 📝 Executive Summary

### Project: AML Detection Pipeline

---

**Business Problem:**  
Financial institutions face two costly problems: missing real money laundering cases (leading to large fines) and generating too many false alerts (overwhelming analysts and causing missed regulatory deadlines).

---

**Our Solution – A 5-Phase Machine Learning Pipeline:**

| Phase | What We Did | Why |
|-------|-------------|-----|
| 1. Preprocessing | Cleaned data, encoded transaction types, scaled numbers | Raw data is not usable by ML models as-is |
| 2. PCA | Reduced many features into a smaller set of principal components | Removes noise and correlated information |
| 3. K-Means | Grouped transactions into behavioral segments | Adds a new, powerful feature based on transaction patterns |
| 4. XGBoost | Trained a gradient boosting classifier to flag fraud | High accuracy on imbalanced data, industry standard |
| 5. Continuous Learning | Updated the model on new monthly data without retraining | Adapts to evolving fraud tactics without storing old data |

---

**Why Not Accuracy?**  
The dataset has <1% fraud. A model predicting "no fraud" for everything would achieve 99%+ accuracy but catch zero criminals. We use **Recall** (did we catch all the fraud?) and **F1-Score** (balance between catching fraud and not annoying analysts with false alarms).

---

**Business Impact:**  
This pipeline reduces false positives, improves analyst efficiency, and continuously adapts — helping the bank meet regulatory deadlines while reducing financial crime risk.

---
## 🎤 Defense Questions & Short Answers

---

**Q1: Why did you use PCA?**  
> Some features in financial data are correlated (e.g., old balance and new balance move together). PCA compresses these into independent components, reduces noise, and speeds up training. We kept enough components to explain 95% of the variance.

---

**Q2: How did you choose the number of PCA components?**  
> We plotted the cumulative explained variance and chose the number of components at the point where we exceed 95%. This is a standard, justified threshold.

---

**Q3: How did you choose the number of clusters (k)?**  
> We used the Elbow Method — we plotted inertia (how tight clusters are) for k=2 to 10, and picked the k where the curve stops dropping sharply (the "elbow"). That's the point of diminishing returns.

---

**Q4: Why not use accuracy to evaluate the model?**  
> Because the dataset is highly imbalanced — less than 1% of transactions are fraud. A model that always predicts "not fraud" would get 99%+ accuracy but be completely useless. Recall tells us how many real fraud cases we caught, and F1-Score balances catching fraud vs. false alarms.

---

**Q5: Why is Recall important in AML?**  
> Missing a fraud case (false negative) means a criminal isn't caught — the bank could face huge regulatory fines. So we want Recall to be high. We're willing to accept some false alarms (false positives) because those just require an analyst to review a transaction.

---

**Q6: What is continuous learning and why does it matter?**  
> Fraudsters change their behavior over time. If we only train once, the model becomes outdated. Continuous learning means we update the model when new data arrives, without using old historical data — so it stays current without growing in complexity.

---

**Q7: Why didn't you retrain from scratch with both old and new data?**  
> It's forbidden by the project rules — and also impractical in real life. Old data would need to be stored forever, and retraining takes a long time. Partial training (warm-start) is faster, private, and sufficient.

---

**Q8: What does scale_pos_weight do in XGBoost?**  
> It tells the model to weight fraud cases more heavily during training. Since fraud is rare, without this, the model would mostly learn to predict "normal" all the time. Setting it to (normal count / fraud count) balances the training signal.

---

**Q9: What is the behavioral_segment feature and why is it useful?**  
> It's a cluster label assigned to each transaction by K-Means. Transactions in the same cluster have similar behavior patterns. Fraud often clusters separately from normal transactions, so this label gives XGBoost an extra behavioral signal that raw features alone don't capture.

---

**Q10: What would you improve if you had more time?**  
> I would add cross-validation for hyperparameter tuning, test SMOTE oversampling to better handle class imbalance, and deploy the model as a REST API so it can receive live transactions and return fraud scores in real time.

---